Evaluate VA/KDI models and RECODe models in AoU

# Setup

In [ ]:
library(tidyverse)
library(data.table)
library(survival)
library(tictoc)
library(mice)
library(devtools)
library(fastcmprsk)
library(timeROC)
library(survAUC)
library(prodlim)
library(DiagrammeR)
library(DiagrammeRsvg)
library(rsvg)
library(pracma)
library(cmprsk)
library(ggrepel)
library(conflicted)
library(riskRegression)

conflict_prefer("tic", "tictoc")
conflict_prefer("toc", "tictoc")
conflict_prefer("filter", "dplyr")
conflict_prefer("lag", "dplyr")

In [ ]:
## Upload file using File -> Open -> Upload, then save to bucket with code below
#system(paste0("gsutil cp ./KDI_model_coefficients_E12202024.txt ", 
#              Sys.getenv('WORKSPACE_BUCKET'), "/data/Feb2025/"), intern=T)

In [ ]:
#Start time
now()

In [ ]:
# load shared functions script
source("functions.R")

In [ ]:
# The date the outcomes file was generated - update as needed
outcomes_date <- "20250414"

In [ ]:
tic("Time to run script")

## Read in Coefficients and covariate data

In [ ]:
KDI_coefs <- fread("KDI_model_coefficients_E12202024.txt") %>%
    mutate(predictor = gsub("Antidepressant_Rx", "Antipsychotics", predictor)) # fix this variable name
    
KDI_predictors <- KDI_coefs %>% distinct(predictor) 
KDI_predictors$predictor

In [ ]:
covars_all <- read_from_bucket("covariates_wide_all_participants.csv", skip_copy=T)

In [ ]:
# Start a cohort generation flowchart to resemble the one in KDI - add rows throughout this script
flowchart_dat <- data.frame(step = "Initial", n = nrow(covars_all))
flowchart_dat

In [ ]:
covars_all <- covars_all %>% 
    filter(!is.na(date_of_birth) & !is.na(gender)) %>%
    filter(gender %in% c("Female", "Male"))

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Demographic filters: No missing age or gender", 
                                 n = nrow(covars_all)))
flowchart_dat

In [ ]:
DMrecs <- read_from_bucket("DM_allrecords.csv", skip_copy=T)

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients with 1+ Diabetes Code",
                                 n = length(unique(DMrecs$person_id))))

# Get outcomes and time-invariant covariates, define cohort

In [ ]:
DMcohort <- read_from_bucket(paste0("outcomes_diabetics_", outcomes_date, ".csv"), skip_copy = TRUE) %>%
    inner_join_quiet(covars_all)
nrow(DMcohort)

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients meeting criteria for T1D", 
                                 n = DMcohort %>% filter(DM_type == "T1D") %>% nrow())) 
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients meeting criteria for T2D", 
                                 n = DMcohort %>% filter(DM_type == "T2D") %>% nrow())) 
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients meeting criteria for T1D or T2D", 
                                 n = DMcohort %>% filter(DM_type %in% c("T1D", "T2D")) %>% nrow())) 

In [ ]:
 DMcohort <- DMcohort %>%
    dplyr::rename(
                 DmDx_first = DMDx_first,
                 SmokingFactor = smoke_category,
                 Ethnicity = ethnicity,
                 Race = race,
                 Gender = gender,
                 DeathDateTime = death_date,
                 BMI_survey = BMI) %>%
    mutate(Age_DmDx = round(decimal_date(DmDx_first) - decimal_date(as.Date(date_of_birth)), 1)) %>%
    filter(Age_DmDx >= 21 & DM_type %in% c("T1D", "T2D"))

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients diagnosed with DM after age 21", 
                                 n = DMcohort %>% nrow()))
flowchart_dat

In [ ]:
 DMcohort <- DMcohort %>%
    filter(DM_prior2yr_outpat == 1) %>%
    select(person_id, DmDx_first, Age_DmDx, SmokingFactor, Ethnicity, Race, Gender, CVD2, years_dm_to_CVD2,
           RenalFailure, RenalFailure_first, years_dm_to_RenalFailure,
           DM_type, AveragePain7Days, BMI_survey, DeathDateTime, years_from_dm_to_censor, 
           last_outpat_or_home_visit, last_visit_any, MilitaryHealthInsurance, KidneyTransplant_survey) %>%
    # Make dates in relation to the year 2000, same as in KDI so that functions work as-is
    mutate(DmDx_first = decimal_date(DmDx_first) - 2000,
          RenalFailure_first = decimal_date(RenalFailure_first) - 2000,
          DeathDateTime = decimal_date(DeathDateTime) - 2000, 
          last_outpat_or_home_visit = decimal_date(last_outpat_or_home_visit) - 2000,
          last_visit_any = decimal_date(last_visit_any) - 2000)

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients with 2 years of outpatient data prior to first DM code/med", 
                                 n = DMcohort %>% nrow()))

In [ ]:
DMcohort <- DMcohort %>% 
    mutate(Race = case_when(Race %in% c(NA, "More than one population", "Middle Eastern or North African",
                                       "None of these") ~ "OTHER_UNKNOWN", 
                            TRUE ~ Race), 
           Ethnicity = case_when(Ethnicity %in% c(NA, "None Of These") ~ "OTHER_UNKNOWN", 
                                 TRUE ~ Ethnicity)
          )
table(DMcohort$Race)
table(DMcohort$Ethnicity)

In [ ]:
possible_VA_users <- DMcohort %>%
    filter(MilitaryHealthInsurance == 1)

# We expect the majority to be male
mean(possible_VA_users$Gender == "Male")

In [ ]:
DMcohort <- DMcohort %>%
    filter(is.na(MilitaryHealthInsurance) | MilitaryHealthInsurance == 0)

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients without VA/Military Health Insurance", 
                                 n = DMcohort %>% nrow()))

In [ ]:
DMcohort_exclude_unkown_transplant <- DMcohort %>%
    filter(KidneyTransplant_survey == 1 & RenalFailure == 0)

nrow(DMcohort_exclude_unkown_transplant) # self-report transplant but not in EHR

In [ ]:
DMcohort <- DMcohort %>%
    anti_join_quiet(DMcohort_exclude_unkown_transplant)

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Patients without self-reported kidney transplant with unknown date", 
                                 n = DMcohort %>% nrow()))
flowchart_dat

In [ ]:
DMcoh_ids <- DMcohort %>% distinct(person_id)
nrow(DMcoh_ids)

# Align time-dependent biomarker data

In [ ]:
# set time limits
years_prior_med = 0.1
years_prior_biomarker = 2
years_prior_other = 99

In [ ]:
# Need to read everything in before imputation. List datasets to be loaded
biomarkers <- c(
    'ALT', 'AST', 'Albumin_BSP', 'Alkaline_Phosphatase',  'BMI', 'BUN', 'Basophil_count', 
    'Basophil_fraction', 'Bicarbonate_bsp', 'Bilirubin_UrineStrip', 'Bilirubin_bsp_conjugated', 
    'Bilirubin_bsp_total', 'CRP', 'Calcium_bsp', 'Chloride_bsp', 'DBP', 'Eosinophil_count', 
    'Eosinophil_fraction', 'Glucose', 'HDL', 'HbA1c', 'HeartRate', 'INR', 'LDL', 'Lymphocyte_count', 
    'LymphFra', 'MCH', 'MCHC', 'MCV', 'MPV', 'Magnesium_bsp', 'Neutrophil_count',  
    'Neutrophil_fraction', 'Pain', 'Potassium', 'Prothrombin_time',  'RBC', 'RDW_ratio', 'SBP', 
    'SaO2', 'Sodium_bsp', 'TotalCK', 'Troponin', 'UACR', 'Uric_Acid', 'WBC', 'eGFR_creatinine',  
    'hematocrit', 'hemoglobin',  'platelets',  'total_cholesterol', 'triglycerides')

# meds
medications <- c(
    'ACEInhibitors','AngiotensinIIRB', 'Anticoagulants', 'DPP4', 'Diuretics','GLP1_INJECTED', 
    'Antipsychotics', 'Aspirin', 'BetaBlockers','CalciumChannelBlockers', 'Corticosteroids_systemic', 
    'ProtonPumpInhibitors', 'SGLT2', 'Sulfonylurea','Thiazoladinedione', 'glucagon','insulin', 
    'lipids_lowering_agents_v2', 'metformin',  'Other_oraldm', "OpioidForPain_Rx")

# other
other_longitudinal_covars <- c('CCI_time_dependent', 'DCSI_time_dependent')

Apply the inclusion criteria: Have to have 1+ A1C, Serum Creatinine, and LDLC in the relevant time frame. 

In [ ]:
for (bm in c("HbA1c", "LDL")){
    tmp <- read_from_bucket(paste0("bm_", bm, ".csv"), skip_copy = T) %>% 
        yr_round(.) 
    names(tmp)[3] <- bm
    assign(bm, tmp) 
    rm(tmp)
}

In [ ]:
# Require HbA1c
DMcohort <- DMcohort %>%
    inner_join_quiet(HbA1c %>% distinct(person_id))
DMcoh_ids <- DMcoh_ids %>%
    inner_join_quiet(HbA1c %>% distinct(person_id))

Read in eGFR; there are 2 variables of interest stored in the file, eGFR and creatinine

In [ ]:
EGFR <- read_from_bucket("bm_eGFR_creatinine.csv", skip_copy = TRUE) 
EGFR <- EGFR %>% inner_join_quiet(DMcoh_ids) %>% 
    mutate(measurement_date = decimal_date(measurement_date) - 2000)
EGFR <- align_date_dm(EGFR, marker_names = c("eGFR", "Creatinine"), years_prior = years_prior_biomarker) %>%
    dplyr::rename(EGFR = eGFR, yrs = t)

In [ ]:
# require Serum Creatinine
DMcohort <- DMcohort %>%
    inner_join_quiet(EGFR %>% distinct(person_id))
DMcoh_ids <- DMcoh_ids %>%
    inner_join_quiet(EGFR %>% distinct(person_id))
HbA1c <- inner_join_quiet(HbA1c, DMcoh_ids)

In [ ]:
# Require LDL
DMcohort <- DMcohort %>%
    inner_join_quiet(LDL %>% distinct(person_id))
DMcoh_ids <- DMcoh_ids %>%
    inner_join_quiet(LDL %>% distinct(person_id))

HbA1c <- inner_join_quiet(HbA1c, DMcoh_ids)
EGFR <- inner_join_quiet(EGFR, DMcoh_ids)

In [ ]:
flowchart_dat <- rbind(flowchart_dat,
                      data.frame(step = "Has at least 1 measurement each of LDL, HbA1c, and serum Creatinine at/after DmDx", 
                                 n = DMcohort %>% nrow()))
flowchart_dat

In [ ]:
# flowchart data
fc_dat <- flowchart_dat %>%
  mutate(stepno = row_number()) %>%
  mutate(txt = paste0(step, "\n", format(n, big.mark = ",", scientific=F)))
fc_dat

In [ ]:
flowchart <- DiagrammeR::grViz(paste0("
                  digraph flowchart {
                  graph [layout = dot, rankdir = TB]
                  
                  node [shape = box, style = filled, fillcolor = white]
                  
                  A [label = '", fc_dat$txt[1], "']
                  B [label = '", fc_dat$txt[2], "']
                  C [label = '", fc_dat$txt[3], "']
                  D [label = '", fc_dat$txt[4], "']
                  E [label = '", fc_dat$txt[5], "']
                  F [label = '", fc_dat$txt[6], "']
                  G [label = '", fc_dat$txt[7], "']
                  H [label = '", fc_dat$txt[8], "']
                  I [label = '", fc_dat$txt[9], "']
                  J [label = '", fc_dat$txt[10], "']
                  K [label = '", fc_dat$txt[11], "']
                  
                  A -> B -> C 
                  C -> D
                  C -> E
                  D -> F
                  E -> F
                  F -> G -> H -> I -> J -> K
                  }
                  "))
flowchart 

In [ ]:
DiagrammeRsvg::export_svg(flowchart) %>%
    charToRaw() %>%
    rsvg_png("flowchart_AOU.png", width=1300, height=1500)

Read and processes the rest of the biomarkers

In [ ]:
tic("Total time to read in and process biomarkers")
for (bm in biomarkers[!(biomarkers %in% c("HbA1c", "LDL", "eGFR_creatinine", "BMI", "UACR"))]){
    tic(bm)
    if (!exists(bm)) {
        tmp <- read_cols(paste0("bm_", bm, ".csv"), select = c("person_id", "measurement_date", "value_as_number"), 
                         skip_copy = T) %>%
            yr_round(.) 
        names(tmp)[3] <- bm
        assign(bm, tmp)    
        rm(tmp)
        }
    toc()
}
toc()

A couple of datasets that had extra processing must be read in separately. 

In [ ]:
BMI <- read_from_bucket("bm_BMI.csv", skip_copy = TRUE) %>%
    dplyr::rename(value_as_number = BMI) %>%
    yr_round(.) %>%
    dplyr::rename(BMI = value_as_number)

In [ ]:
UACR <- read_from_bucket("bm_UACR.csv", skip_copy = TRUE) %>%
    dplyr::rename(value_as_number = UACR) %>%
    yr_round(.) %>%
    dplyr::rename(UACR = value_as_number)

Map the KDI biomarker names (generally, "ShortNames") to the ones assigned in AoU (intended to be more descriptive, for general use), if they differ. 

In [ ]:
names_change <- data.frame(
    KDIname = c('HDLC', 'LDLC', 'Cl', 'EosFra', 'Eos', 
               'LymphFra',  'Lymph', 
              'AlkalinePhosphatase', 'Platelet', 'Pulse', 'Trig', 'A1C',
              'TotChol', 'BasoFra', 'Baso', 'Mg',
               'Na', 'Ca', 'PT',
              'Bicarbonate', 'Hematocrit', 'NeutFra', 'Neut',
              'PO2', 'Albumin', 'RDW', 'Bilirubin_BSP_conjugated', 'Uric_Acid_BSP',
               'Bilirubin_BSP_total', 'BilirubinStick', 'Creat_BSP', 'Hemoglobin', 'K', 
              'CChannelBlockers', 'ProtonPumpInhib', 'Corticosteroids'), 
    AoUname = c('HDL',  'LDL', 'Chloride_bsp', 'Eosinophil_fraction', 'Eosinophil_count', 
               'LymphFra', 'Lymphocyte_count', 
               'Alkaline_Phosphatase', 'platelets', 'HeartRate', 'triglycerides', 'HbA1c',
              'total_cholesterol', 'Basophil_fraction', 'Basophil_count', 'Magnesium_bsp',
               'Sodium_bsp', 'Calcium_bsp', 'Prothrombin_time',
              'Bicarbonate_bsp', 'hematocrit', 'Neutrophil_fraction', 'Neutrophil_count',
              'SaO2', 'Albumin_BSP', 'RDW_ratio', 'Bilirubin_bsp_conjugated', 'Uric_Acid',
               'Bilirubin_bsp_total', 'Bilirubin_UrineStrip', 'Creatinine', 'hemoglobin', 'Potassium',
               'CalciumChannelBlockers', 'ProtonPumpInhibitors', 'Corticosteroids_systemic')
)
names_change

In [ ]:
write_to_bucket(names_change, "AoU_varname_KDI_map.csv")

# Align time-dependent conditions, procedures, medications, etc

In [ ]:
load_RDATA_from_bucket('timedep_conditions_and_procedures.RData')

In [ ]:
CondProc_objs <- c(
    "doppler",
    "Backpain",
    "EH_COMDIAB",
    "EH_RENAL",
    "EH_ARRHYTH",
    "EH_HYPERTENS",
    "ODEPRdx_poss",
    "EH_CHRNPULM",
    "EH_LIVER",
    "EH_ELECTRLYTE",
    "EH_OBESITY",
    "Proteinuria_Dx",
    "SleepDisorder")

In [ ]:
# process each condition/procedure
for (cond in CondProc_objs) {
    tmp <- get(cond)$first %>% inner_join_quiet(DMcoh_ids) %>%
        select(-contains("concept"))
    names(tmp)[2:3] <- c("measurement_date", "value_as_number")
    tmp <- tmp %>% mutate(measurement_date = decimal_date(measurement_date) -2000) %>% 
        align_date_dm(., years_prior = years_prior_other)
    assign(cond, tmp)
}

In [ ]:
# Read in medications and filter to current cohort
tic("Total time to load medications")
for (med in medications) {
    tmp <- read_from_bucket(paste0(med, ".csv")) %>% 
        inner_join_quiet(DMcoh_ids) %>%
        left_join_quiet(DMcohort %>% select(person_id, DeathDateTime, DmDx_first)) %>%
        dplyr::rename(measurement_date = drug_exposure_start_datetime) %>% 
        mutate(measurement_date = decimal_date(as.Date(measurement_date)) - 2000) %>%
        filter(measurement_date <= DeathDateTime | is.na(DeathDateTime)) %>%
        mutate(yrs = measurement_date - DmDx_first,
              yrs_rounded = round(yrs, 1)) %>%
        filter(yrs >= -1) %>%
        select(-c(DeathDateTime, DmDx_first)) %>%
        mutate(refills = coalesce(refills, 0))
    
    assign(med, tmp)
}
toc()

In [ ]:
# We updated this file, rename object and keep code the same from here
lipids_lowering_agents <- lipids_lowering_agents_v2
rm(lipids_lowering_agents_v2)

In [ ]:
# Break lipids drugs into subcategories
Niacin <- lipids_lowering_agents %>% filter(Niacin)
Ezetimibe <- lipids_lowering_agents %>% filter(Ezetimibe)
Fibrates <- lipids_lowering_agents %>% filter(Fibrates)
BAS <- lipids_lowering_agents %>% filter(BAS)
other_lipid_lowering <- lipids_lowering_agents %>%
   filter(OtherLipidLowering | FattyAcid)

Statin_High <- lipids_lowering_agents %>% filter(Statin & statin_dose_class == "High")
Statin_Medium  <- lipids_lowering_agents %>% filter(Statin & statin_dose_class == "Medium")
Statin_Low  <- lipids_lowering_agents %>% filter(Statin & statin_dose_class == "Low")

In [ ]:
# Use subcategories
medications <- medications[!(medications %in% c("lipids_lowering_agents", "lipids_lowering_agents_v2"))]
medications <- c(medications, "Statin_High", "Statin_Medium", "Statin_Low", "Niacin", "Ezetimibe", "BAS", "Fibrates")

We use the duration of medication usage with the prescriptions (days supply, quantity).  A large quantity of records are missing these values, so we use the typically observed days supply and/or quantity, where possible.

In [ ]:
medsumm <- data.frame(n=-1)
for (med in medications) {
    medsumm.x1 <- get(med) %>%
        group_by(quantity) %>%
        count() %>%
        mutate(var = "quantity") %>%
        dplyr::rename(value = quantity)
    medsumm.x2 <- get(med) %>%
        group_by(refills) %>%
        count() %>%
        mutate(var = "refills")  %>%
        dplyr::rename(value = refills)
    medsumm.x3 <- get(med) %>%
        group_by(days_supply) %>%
        count() %>%
        mutate(var = "days_supply")  %>%
        dplyr::rename(value = days_supply)
     medsumm.x4 <- get(med) %>%
        mutate(q_per_day = quantity/days_supply) %>%
        group_by(q_per_day) %>%
        count() %>%
        mutate(var = "q_per_day")  %>%
        dplyr::rename(value = q_per_day)
    medsumm.x <- full_join_quiet(medsumm.x1, medsumm.x2) %>%
        full_join_quiet(medsumm.x3) %>%
        full_join_quiet(medsumm.x4) %>%
        mutate(med)
    medsumm <- full_join_quiet(medsumm, medsumm.x)
}

medsumm <- medsumm %>% filter(n != -1)

In [ ]:
# Quantity: visualize
medsumm %>%
    filter(var == "quantity") %>%
    filter(value < 120 & value >= 0) %>%
    ggplot(aes(x=value, y = n, color = med)) + 
    geom_bar(stat = "identity", position="dodge") +
    facet_wrap(~med, scales = "free_y") + 
    theme(legend.position = "none")

In [ ]:
# Days Supply: visualize
medsumm %>%
    filter(var == "days_supply") %>%
    filter(value < 120 & value >= 0) %>%
    ggplot(aes(x=value, y = n, color = med)) + 
    geom_bar(stat = "identity", position="dodge") +
    facet_wrap(~med, scales = "free_y", ncol=5) + 
    theme(legend.position = "none")

In [ ]:
# Quantity per day supply: visualize
medsumm %>%
    filter(var == "q_per_day") %>%
    filter(value < 5 & value >= 0) %>%
    ggplot(aes(x=value, y = n, color = med)) + 
    geom_bar(stat = "identity", position="dodge") +
    facet_wrap(~med, scales = "free_y") + 
    theme(legend.position = "none")

In [ ]:
# fill in quantity, days supply with typical values where only one of the two is missing
for (med in medications){
    tmp <- get(med) 
    
    if (med %in% c("metformin", "BAS")) {
        tmp <- tmp %>% mutate(days_supply = coalesce(days_supply, quantity/2)) %>% 
            mutate(quantity = coalesce(quantity, days_supply*2))
    } else if (med == "other_oraldm") {
        tmp <- tmp %>% mutate(days_supply = coalesce(days_supply, quantity/3)) %>%
            mutate(quantity = coalesce(quantity, days_supply*3))
    } else {
         tmp <- tmp %>% mutate(days_supply = coalesce(days_supply, quantity)) %>%
            mutate(quantity = coalesce(quantity, days_supply))
    }
    assign(med, tmp)
}

In [ ]:
# Visualize again after filling in single missing values
medsumm <- data.frame(n=-1)
for (med in medications) {
    medsumm.x1 <- get(med) %>%
        group_by(quantity) %>%
        count() %>%
        mutate(var = "quantity") %>%
        dplyr::rename(value = quantity)
    medsumm.x2 <- get(med) %>%
        group_by(refills) %>%
        count() %>%
        mutate(var = "refills")  %>%
        dplyr::rename(value = refills)
    medsumm.x3 <- get(med) %>%
        group_by(days_supply) %>%
        count() %>%
        mutate(var = "days_supply")  %>%
        dplyr::rename(value = days_supply)
     medsumm.x4 <- get(med) %>%
        mutate(q_per_day = quantity/days_supply) %>%
        group_by(q_per_day) %>%
        count() %>%
        mutate(var = "q_per_day")  %>%
        dplyr::rename(value = q_per_day)
    medsumm.x <- full_join_quiet(medsumm.x1, medsumm.x2) %>%
        full_join_quiet(medsumm.x3) %>%
        full_join_quiet(medsumm.x4) %>%
        mutate(med)
    medsumm <- full_join_quiet(medsumm, medsumm.x)
}

medsumm <- medsumm %>% filter(n != -1)

In [ ]:
medsumm %>%
    filter(var == "days_supply" & value < 120 & value >= 0) %>%
    ggplot(aes(x=value, y = n, color = med)) + 
    geom_bar(stat = "identity", position="dodge") +
    facet_wrap(~med, scales = "free_y", ncol=5) + 
    theme(legend.position = "none")

In [ ]:
medsumm %>%
    filter(var == "quantity" & value < 120 & value >= 0) %>%
    ggplot(aes(x=value, y = n, color = med)) + 
    geom_bar(stat = "identity", position="dodge") +
    facet_wrap(~med, scales = "free_y", ncol=5) + 
    theme(legend.position = "none")

In [ ]:
# Fill in where quantity and days supply were both missing - periods < 30 days don't matter due to minimum time interval
for (med in medications){
    tmp <- get(med) 
    
    if (med %in% c('ACEInhibitors', 'AngiotensinIIRB', 'BetaBlockers', 'CalciumChannelBlockers', 
                   'Diuretics', 'Statin_High', 'Statin_Medium', 'Statin_Low', 'Ezetimibe', 
                   'Sulfonylurea', 'metformin', 'Fibrates', 'Niacin', 'ProtonPumpInhibitors',
                   'Aspirin', 'SGLT2', 'DPP4')) {
        tmp <- tmp %>% mutate(days_supply = coalesce(days_supply, 90))
    } else if (med %in% c('Anticoagulants', 'GLP1_INJECTED', 'Antipsychotics', 
                          'Corticosteroids_systemic', 'ProtonPumpInhibitors',  'Thiazoladinedione', 
                          'glucagon', 'insulin', 'Other_oraldm', 'BAS', 'OpioidForPain_Rx')) {
        tmp <- tmp %>% mutate(days_supply = coalesce(days_supply, 30)) 
    }
    assign(med, tmp)
}

In [ ]:
# process the time-dependent medications data into time intervals with 0/1 status
for (med in medications){
    tmp <- med.gap(get(med) %>% 
                   dplyr::rename(drug_exposure_start_datetime = measurement_date,
                                DaysSupply = days_supply))
    assign(med, tmp)
}

In [ ]:
DCSI <- read_from_bucket('DCSI_time_dependent.csv', skip_copy=T)

In [ ]:
DCSI <- align_date_dm(data = DCSI %>% mutate(tstart = decimal_date(tstart) -2000),
                       marker_names = "DCSIscore", date_name = "tstart", years_prior = years_prior_other)

In [ ]:
# Checkpoint time - save image
now()
ls()
save.image("chkpoint1_latest.RData")

In [ ]:
#load("chkpoint1_latest.RData")

# Create tdept and Landmark data, impute

## Data processing, tdept

In [ ]:
data_std <- DMcohort %>%
    filter(years_from_dm_to_censor >= 0 ) %>%
    mutate(years_from_dm_to_censor = round(years_from_dm_to_censor, 1)) %>%
    mutate(death_tte = round(DeathDateTime - DmDx_first, 1),
           death01 = ifelse(!is.na(DeathDateTime), 1, 0)) %>%
    filter(death_tte >= 0 | is.na(death_tte)) %>%
    dplyr::select(-DeathDateTime) %>%
    mutate(years_dm_to_RenalFailure = round(years_dm_to_RenalFailure, 1)) %>% 
    mutate(death_tte = coalesce(death_tte, years_from_dm_to_censor)) %>%
    mutate(event_CR = ifelse(RenalFailure == 1, 1, ifelse(death01 == 1, 2, 0)),
           CR_tte = ifelse(RenalFailure == 1, years_dm_to_RenalFailure, death_tte)) %>%
    filter(CR_tte > 0.01) %>%
    arrange(person_id)

In [ ]:
# ignore deaths (switch to censored) occurring more than 6mo after censoring, as we may not be able to observe renal failure
data_std <- data_std %>%
    mutate(LF_at_death = event_CR == 2 & CR_tte - years_from_dm_to_censor > 0.5,
           event_CR = ifelse(LF_at_death, 0, event_CR),
             CR_tte = ifelse(LF_at_death, years_from_dm_to_censor, CR_tte)) %>%
    select(-LF_at_death)

In [ ]:
data_std %>% count(event_CR) %>% mutate(prop = round(n/sum(n), 3))

Create the initial time-INdependent dataset. 

In [ ]:
tindpt <- tmerge(data_std, data_std, id = person_id, 
                endpt = event(years_dm_to_RenalFailure, RenalFailure),
                death = event(death_tte, death01))

Create the initial time-dependent dataset. 

In [ ]:
tdept <- tmerge(data1 = tindpt, data2 = EGFR, id = person_id, 
                EGFR = tdc(yrs, EGFR), Creatinine = tdc(yrs, Creatinine))

In [ ]:
# Merge another dataset into the existing time-dependent dataset
tdept <- tmerge(tdept, DCSI, id = person_id, DCSI = tdc(t, DCSIscore))

In [ ]:
# list the remaining biomarkers to merge
biomarkers_merge <- biomarkers[!(biomarkers %in% c("eGFR_creatinine"))]
biomarkers_merge

In [ ]:
# tmerge biomarkers
for (bm in biomarkers_merge){
    data2 <- get(bm) %>% inner_join_quiet(DMcoh_ids)
    var_data2 <- names(data2)[3]
    tdept <- tmerge(tdept, data2, id = person_id, newtdc = tdc(yrs, get(var_data2)))
    names(tdept)[ncol(tdept)] <- var_data2
    rm(data2, var_data2, bm)
}

In [ ]:
# tmerge conditions and procedures
for (cond in CondProc_objs){
    data2 <- get(cond) %>% inner_join_quiet(DMcoh_ids)
    tdept <- tmerge(tdept, data2, id = person_id, newtdc = tdc(t))
    names(tdept)[ncol(tdept)] <- cond
    rm(data2, cond)
}

In [ ]:
# tmerge processed medications
for (med in medications){
    data2 <- get(med) %>% inner_join_quiet(DMcoh_ids)
    tdept <- tmerge(tdept, data2, id = person_id, newtdc = tdc(t, med))
    names(tdept)[ncol(tdept)] <- med
    rm(data2, med)
}

Make additional modifications

In [ ]:
# Add derived variables in a new object
tdept_expand <- tdept %>%
    mutate(PulsePressure = SBP - DBP, 
           age_td = Age_DmDx + tstart)

In [ ]:
# Create a key to rename variables to VA names
name_key <- names_change$KDIname
names(name_key) <- names_change$AoUname

In [ ]:
# Rename to KDI names, or keep name if not in key
names(tdept_expand) <- coalesce(name_key[names(tdept_expand)] , names(tdept_expand))

In [ ]:
# new lists of biomarkers and medications, after renaming
biomarkersF <- coalesce(name_key[biomarkers], biomarkers)
medicationsF <- coalesce(name_key[medications], medications)

unique(biomarkersF)
unique(medicationsF)

In [ ]:
# time-updating CVD status
tdept_expand <- tdept_expand %>%
    mutate(CVD2 = ifelse(CVD2 ==1 & years_dm_to_CVD2 <= tstart, 1, 0)) %>%
    select(-years_dm_to_CVD2) 

In [ ]:
# tdept of biomarkers only; then convert to missingness indicators
tdept_expand_bm <- tdept_expand %>% 
    select(person_id, tstart, tstop, EGFR, Creat_BSP, any_of(unique(biomarkersF)))

In [ ]:
# Create indicators for missing biomarker values
tdept_expand_bm <- tdept_expand_bm %>%
    mutate(across(any_of(c("EGFR", "Creat_BSP", unique(biomarkersF))), 
                  function(x) { as.numeric(is.na(x)) }))

names(tdept_expand_bm)[4:ncol(tdept_expand_bm)] <- paste0(names(tdept_expand_bm)[4:ncol(tdept_expand_bm)], "_NA")

In [ ]:
# Add missingness indicators to the object, generate other categorical variables as needed
tdept_expand <- tdept_expand %>% 
    left_join_quiet(tdept_expand_bm) %>%
    mutate(
        Microalbuminuria_30_le_UACR_lt_300 = as.numeric(!is.na(UACR) & UACR >= 30 & UACR < 300),
        Macroalbuminuria_UACR_ge_300 = as.numeric(!is.na(UACR) & UACR >= 300),
        High_Uric_Acid =  as.numeric(!is.na(Uric_Acid_BSP) & 
                                 ( ( Uric_Acid_BSP >= 7.2 & Gender == "Male") |
                                 ( Uric_Acid_BSP >= 6.1  & Gender == "Female" ) ) )
          )

In [ ]:
# Set medication status to 0 if NA
# Use BMI/Pain from survey data ONLY if NA from EHR pull 
tdept_expand <- tdept_expand %>%
    mutate(across(any_of(unique(medicationsF)), function(x) { coalesce(x, 0)})) %>%
     mutate(SmokingFactor = coalesce(SmokingFactor, "UNKNOWN"),
           BMI = coalesce(BMI, BMI_survey),
           Pain = coalesce(Pain, as.numeric(AveragePain7Days)))

## Generating landmark datasets for observed data prior to imputation, descriptive tables

### Landmark Year 1 

#### Generate dataset

In [ ]:
ydat1 <- landmark(tdept_expand, 1) %>%
    mutate(CR_tte_landmark = CR_tte - 1) %>%
    filter(CR_tte_landmark > 0)

nrow(ydat1)

In [ ]:
write_to_bucket(ydat1, "landmark_y1.RenalFailure_v3.txt")

In [ ]:
# Add insurance data for table1
insurance <- covars_all %>%
    inner_join(DMcoh_ids) %>%
    select(person_id, 
           all_of(c('HealthInsurance', 'MilitaryHealthInsurance', 'Uninsured', 'PrivateHealthInsurance', 
                  'OtherHealthInsurance', 'Medicare', 'Medicaid'))) %>%
    mutate(across(HealthInsurance:Medicaid, as.character))

#### Generate descriptive tables for export defining cohort characteristics at L1

In [ ]:
table1_start <- ydat1 %>% 
    left_join_quiet(insurance %>% select(-MilitaryHealthInsurance)) %>%
    select(-c(AveragePain7Days, BMI_survey, last_outpat_or_home_visit, last_visit_any, KidneyTransplant_survey, 
              death, endpt, tstart, tstop)) %>%
    select(-ends_with("_NA")) %>%
    dplyr::rename(
           `Years to Event (or Censoring)` = CR_tte_landmark,
            `Years to Censoring` = years_from_dm_to_censor,
            `Death` = death01, 
           `Years to Death` = death_tte)  %>%
    mutate(across(!c(BilirubinStick), factorize.i)) %>%
    group_by(event_CR)

In [ ]:
# Table 1 headers
table1_addendum <- table1_start %>%
    select(person_id, event_CR) 
table1_addendum <- table1_addendum %>%
    mutate(event_CR = 99) %>%
    rbind(table1_addendum) %>%
    ungroup() %>%
    mutate(n.total = length(unique(person_id))) %>%
    group_by(event_CR) %>%
    summarize(`0.colname_addendum` = paste0(n(), " (", signif(100*n()/unique(n.total), 3), "%)")) %>%
    t() %>% data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    filter(variable != "event_CR") %>%
    mutate(stat_type = "0.table_colstats")
names(table1_addendum)[2:5] <- c("No Events", "Renal Failure", "Death without Renal Failure", "Overall")
table1_addendum

In [ ]:
# Table 1 median (IQR) values
table1_numeric_iqr <- table1_start %>%
    select_if(is.numeric)
table1_numeric_iqr <- table1_numeric_iqr %>%
    mutate(event_CR = 99) %>%
    rbind(table1_numeric_iqr) %>%
    summarize_all(IQR) %>%
    t() %>% data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    mutate(stat_type = "1.median_iqr") %>%
    filter(!(variable %in% c("event_CR", "DmDx_first", "person_id")))
names(table1_numeric_iqr)[2:5] <- c("No Events", "Renal Failure", "Death without Renal Failure", "Overall")

In [ ]:
# Table 1 n (%) NA values
table1_numeric_NA <- table1_start %>%
    select_if(is.numeric)
table1_numeric_NA <- table1_numeric_NA %>%
    mutate(event_CR = 99) %>%
    rbind(table1_numeric_NA) %>%
    summarize_all(summcat_mask20_NAs) %>%
    t() %>% data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    mutate(stat_type = "2.report_NAs") %>%
    filter(!(variable %in% c("event_CR", "DmDx_first", "person_id")))
names(table1_numeric_NA)[2:5] <- c("No Events", "Renal Failure", "Death without Renal Failure", "Overall")

In [ ]:
# list of variables to count the repeated measures
to_count <- table1_start %>% 
    select_if(is.numeric) %>%
    ungroup() %>% 
    select(-c("Age_DmDx", "Years to Event (or Censoring)", "Years to Censoring", "Years to Death",
              "event_CR", "DmDx_first", "person_id", "age_td", 'DCSI', 'RenalFailure_first', 
              'years_dm_to_RenalFailure', 'CR_tte', any_of(c(CondProc_objs, medications)))) %>% 
    names(.)

to_count

In [ ]:
# all names
to_count_real <- data.frame(KDIname = to_count) %>%
    left_join_quiet(names_change) %>%
    mutate(obj = coalesce(AoUname, KDIname)) %>%
    .$obj
to_count_real

#### Checkpoint 2

In [ ]:
#Time at checkpoint
now()
ls()
save.image("chkpoint2_latest.RData")

In [ ]:
#load("chkpoint2.RData")

#### More tables

Count the number of repeated measures and add to table

In [ ]:
Creatinine <- EGFR # number of measures will be the same
PulsePressure <- SBP # number of measures will be the same

init <- data.frame(person_id = 0) %>% filter(is.na(person_id))

# Add summary for each marker
for (marker in to_count_real){
    suppressMessages(d <- get_n_measures(marker))
    suppressMessages(init <- full_join(init, d))
}

In [ ]:
table1_numeric_nmeasures <- init %>%
    left_join_quiet(table1_start %>% select(person_id, event_CR)) %>%
    group_by(event_CR) 
table1_numeric_nmeasures <- table1_numeric_nmeasures %>%
    mutate(event_CR = 99) %>%
    rbind(table1_numeric_nmeasures) %>%
    summarize_all(function(x){paste0(median(x), " (", q25(x), " - ", q75(x), ")")}) %>%
    t() %>% data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    mutate(stat_type = "3.report_n_measures_y1p") %>%
    filter(!(variable %in% c("event_CR", "DmDx_first", "person_id")))
names(table1_numeric_nmeasures)[2:5] <- c("No Events", "Renal Failure", "Death without Renal Failure", "Overall")

In [ ]:
# table 1 categorical variables
table1_categorical <- table1_start %>%
    select(where(is.factor) | where(is.character), event_CR)  %>%
    mutate_if(is.character, as.factor)
table1_categorical <- table1_categorical %>%
    mutate(event_CR = 99) %>%
    rbind(table1_categorical) %>%
    pivot_longer(!event_CR, names_to = "variable") %>%
    mutate(value = as.character(value)) %>%
    group_by(event_CR, variable) %>%
    mutate(n = n()) %>%
    group_by(event_CR, variable, value) %>% 
    reframe(smry = paste0(n(), " (", signif( (n()/n)*100, 3) , "%)"), n.val = n(), n.tot = unique(n)) %>%
    mutate(smry = ifelse(n.val < 20, paste0("<=20 (<=", signif((20/n.tot)*100, 3), "%)") , smry)) %>% 
    select(-n.val, -n.tot) %>% 
    distinct() %>%
    filter(value != "0") %>%
    mutate(stat_type = "1.report_categories") %>%
    pivot_wider(names_from = "event_CR", values_from = "smry")
names(table1_categorical)[4:7] <- c("No Events", "Renal Failure", "Death without Renal Failure", "Overall")

In [ ]:
table1_everything <- full_join_quiet(table1_numeric_iqr, table1_numeric_NA) %>%
    full_join_quiet(table1_categorical) %>%
    full_join_quiet(table1_numeric_nmeasures) %>% 
    full_join_quiet(table1_addendum) %>%
    arrange(variable, stat_type, value) %>%
    select(variable, stat_type, value, everything())
table1_everything %>% head()

In [ ]:
# Masked rows
table1_everything[grepl("<=20", table1_everything$Overall), ]

In [ ]:
# mask NOEVENT counts where there are masked cells so that the number cannot be derived
table1_everything <- table1_everything %>%
    mutate(`No Events` = ifelse( ( (as.numeric(grepl("<=20", Overall)) +
                            as.numeric(grepl("<=20", `Renal Failure`)) + 
                            as.numeric(grepl("<=20", `Death without Renal Failure`)) ) == 1) & 
                                !grepl("<=20", `No Events`),
                           "Masked", `No Events`))
table1_everything %>% filter(`No Events` == "Masked")

In [ ]:
write_to_bucket(table1_everything, "table1_y1_landmark_v3.csv")

 Also report biomarker summaries by sex instead of status

In [ ]:
# Table headers for biomarkers by sex
table1_sex_addendum <- table1_start %>%
    select(person_id, Gender, event_CR) 
table1_sex_addendum <- table1_sex_addendum %>%
    mutate(Gender = "Overall") %>%
    rbind(table1_sex_addendum) %>%
    ungroup() %>%
    mutate(n.total = length(unique(person_id))) %>%
    group_by(Gender) %>%
    summarize(`0.colname_addendum` = paste0(n(), " (", signif(100*n()/unique(n.total), 3), "%)")) %>%
    t() %>% data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    filter(variable != "Gender") %>%
    mutate(stat_type = "0.table_colstats")
names(table1_sex_addendum)[2:4] <- c("Female", "Male", "Overall")

In [ ]:
# median (IQR) values by sex
table1_numeric_iqr_sex <- table1_start %>%
    select_if(is.numeric) %>% 
    left_join_quiet(table1_start %>% select(person_id, Gender)) 
table1_numeric_iqr_sex <- table1_numeric_iqr_sex %>%
    mutate(Gender = "Overall") %>%
    rbind(table1_numeric_iqr_sex) %>%
    group_by(Gender) %>%
    summarize_all(IQR) %>%
    t() %>% 
    data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    mutate(stat_type = "1.median_iqr") %>%
    filter(!(variable %in% c("event_CR", "DmDx_first", "person_id", "Gender")))
names(table1_numeric_iqr_sex)[2:4] <- c("Female", "Male", "Overall")

In [ ]:
# initialize
init_sex <- data.frame(person_id = 0) %>% filter(is.na(person_id))

# summarize for each marker
for (marker in to_count_real){
    suppressMessages(d_sex <- get_n_measures_sex(marker))
    suppressMessages(init_sex <- full_join(init_sex, d_sex))
}

In [ ]:
# Count number of repeated measures by sex
table1_numeric_nmeasures_sex <- init_sex 

table1_numeric_nmeasures_sex <- table1_numeric_nmeasures_sex %>%
    mutate(Gender = "Overall") %>%
    rbind(table1_numeric_nmeasures_sex) %>%
    group_by(Gender) %>%
    summarize_all(function(x){paste0(median(x), " (", q25(x), " - ", q75(x), ")")}) %>%
    t() %>% data.frame() %>%
    tibble::rownames_to_column() %>%
    dplyr::rename(variable = rowname) %>%
    mutate(stat_type = "3.report_n_measures_y1p") %>%
    filter(!(variable %in% c("Gender", "person_id")))
names(table1_numeric_nmeasures_sex)[2:4] <- c("Female", "Male", "Overall")

In [ ]:
biomarkers_by_sex <- rbind(table1_numeric_nmeasures_sex,
                           table1_numeric_iqr_sex) %>%
    full_join_quiet(table1_sex_addendum) %>%
    arrange(variable, stat_type)
biomarkers_by_sex

In [ ]:
write_to_bucket(biomarkers_by_sex, "biomarker_summaries_by_sex_L1_v3.csv")

### Landmark Year 5

In [ ]:
ydat5 <- landmark(tdept_expand, 5) %>%
    mutate(CR_tte_landmark = CR_tte - 5) %>%
    filter(CR_tte_landmark > 0)

nrow(ydat5)

In [ ]:
write_to_bucket(ydat5, "landmark_y5.RenalFailure_v3.txt")

In [ ]:
#ydat5 <- read_from_bucket("landmark_y5.RenalFailure_v3.txt")

In [ ]:
table(ydat5$event_CR)
prop.table(table(ydat5$event_CR))

### Landmark Year 10

In [ ]:
ydat10 <- landmark(tdept_expand, 10) %>%
    mutate(CR_tte_landmark = CR_tte - 10) %>%
    filter(CR_tte_landmark > 0)

nrow(ydat10)

In [ ]:
write_to_bucket(ydat10, "landmark_y10.RenalFailure_v3.txt")

In [ ]:
#ydat10 <- read_from_bucket("landmark_y10.RenalFailure_v3.txt")

In [ ]:
table(ydat10$event_CR)
prop.table(table(ydat10$event_CR))

## Performing imputation at y1

### Imputing

In [ ]:
# Read in the instructions for which markers to impute vs. categorize
imp_key <- fread("imputed_vs_categorized_key_train.csv") %>%
    rbind(fread("imputed_vs_categorized_key_test.csv")) %>%
    distinct(marker_name, form)

In [ ]:
continuous_KDI <- c('Age_DmDx', 'age_td', 
                    imp_key %>% filter(form == "imputed") %>% .$marker_name)
continuous_KDI

#Platelets, triglycerides, and WBC are standardized after log transform
vars_standardized_KDI <- c(continuous_KDI[!(continuous_KDI %in% c("WBC", "Trig", "Platelet"))],
                'Platelet_ln', 'Trig_ln', 'WBC_ln')
vars_standardized_KDI

In [ ]:
continuous_AoU <- continuous_KDI[continuous_KDI %in% names(ydat1)]
not_extracted_AoU <- continuous_KDI[!(continuous_KDI %in% names(ydat1))]
continuous_AoU
not_extracted_AoU

In [ ]:
# Check missingness rates
x <- apply(ydat1[c(continuous_AoU)], 2, function(x) { mean(is.na(x))*100 })
sort(x[x>0]) %>% signif(2)
rm(x)

Bilirubin Stick and PO2 were imputed in KDI but have very high missingness in AoU.  Therefore, use median imputation. 

In [ ]:
median(ydat1$BilirubinStick, na.rm=T)
median(ydat1$PO2, na.rm=T)

In [ ]:
biomarkers_categorized_KDI <- imp_key %>% 
    filter(form == "categorized") %>% .$marker_name
biomarkers_categorized_KDI

biomarkers_categorize_AoU <- biomarkers_categorized_KDI[biomarkers_categorized_KDI %in% names(ydat1)]
biomarkers_categorize_AoU

biomarkers_cat_not_extracted <- setdiff(biomarkers_categorized_KDI, biomarkers_categorize_AoU)
biomarkers_cat_not_extracted 

In [ ]:
y1dat_preimpute <- ydat1 %>%
    mutate(across(everything(), factorize.i)) %>%
    mutate(PO2 = coalesce(PO2, median(ydat1$PO2, na.rm=T)),
           DCSI = coalesce(DCSI, 0), # will be non-zero if there is any record of complications
          BilirubinStick = coalesce(BilirubinStick, median(ydat1$BilirubinStick, na.rm=T))) %>%
    #categorize
    mutate(AST_lt_10 = coalesce(as.numeric(AST < 10), 0),
          AST_ge_40 = coalesce(as.numeric(AST >= 40),  0),
           Bilirubin_BSP_conjugated_ge_0.3 = coalesce(as.numeric(Bilirubin_BSP_conjugated >= 0.3), 0),
          Ca_lt_8 = coalesce(as.numeric(Ca < 8), 0),
          Ca_ge_10.4 = coalesce(as.numeric(Ca >= 10.4), 0),
          CRP_ge_10 = coalesce(as.numeric(CRP >= 10), 0),
          m_1_le_INR_lt_2 = coalesce(as.numeric(INR >= 1 & INR < 2), 0),
           m_2_le_INR_lt_3 = coalesce(as.numeric(INR >= 2 & INR < 3), 0),
           INR_ge_3 = coalesce(as.numeric(INR >= 3), 0),
           Mg_lt_1.6 = coalesce(as.numeric(Mg < 1.6), 0),
           Mg_ge_2.8 = coalesce(as.numeric(Mg >= 2.8), 0),
           PT_lt_10 = coalesce(as.numeric(PT < 10), 0),
           PT_ge_13 = coalesce(as.numeric(PT >= 13), 0),
           TotalCK_ge_200 = coalesce(as.numeric(TotalCK >= 200), 0),
           Troponin_ge_0.4 = coalesce(as.numeric(Troponin >= 0.4), 0),
          ) 


y1dat_preimpute_clean <- y1dat_preimpute %>%
    select(-all_of(c(biomarkers_categorize_AoU))) %>%
    select(-any_of(paste0(continuous_AoU, "_NA"))) %>%
    select(-c(DmDx_first, years_dm_to_RenalFailure, DM_type, years_from_dm_to_censor,
          last_outpat_or_home_visit, last_visit_any, KidneyTransplant_survey, death_tte, 
          death01, event_CR, tstart, tstop, endpt, death, CR_tte_landmark, 
             RenalFailure_first, AveragePain7Days, BMI_survey, MilitaryHealthInsurance))

In [ ]:
# Get the matrix of predictors
pred1 <- quickpred(y1dat_preimpute_clean, 
                   mincor = 0.05, minpuc = 0.01, 
                   exclude = c("person_id", # avoid these vars when imputing. 
                               "age_td")) # collinear with Age_DmDx

In [ ]:
tic("Generating the mice object")
obj_mice <- mice(y1dat_preimpute_clean, m = 1, seed = 12345, pred = pred1)
toc()

In [ ]:
obj_mice$loggedEvents

In [ ]:
saveRDS(obj_mice, "mice.y1.RenalFailure_v3.RDS")

In [ ]:
# save the not-included variables
y1_save_vars <- y1dat_preimpute %>% 
    select(-c(names(y1dat_preimpute_clean %>% select(-person_id)))) %>%
    select(-any_of(biomarkers_categorize_AoU)) %>%
    select(-ends_with("_NA"))

In [ ]:
imputed_y1_prexform <- complete(obj_mice) %>% 
    full_join(y1_save_vars)

In [ ]:
write_to_bucket(imputed_y1_prexform,  "landmark_y1_imputed_prexform.RenalFailure_v3.txt")

### Apply transformations

In [ ]:
vars_standardize_AoU <- vars_standardized_KDI[!(vars_standardized_KDI %in% 
                                                c("Mono", "MonoFra", "Respiration", "Total_Prot"))]

# Apply transformations after imputation:
imputed_y1_postxform <- imputed_y1_prexform %>%
    mutate(Trig_ln = log(Trig),
          Platelet_ln = log(Platelet),
          WBC_ln = log(WBC)) %>%
    mutate(across(all_of(c(vars_standardize_AoU, "DCSI")), standardize)) %>%
    mutate(across(everything(), factorize.i)) %>% 
    select(-Trig, -Platelet, -WBC) %>%
    dplyr::rename(Trig = Trig_ln, Platelet = Platelet_ln, WBC = WBC_ln)

In [ ]:
write_to_bucket(imputed_y1_postxform,  "landmark_y1_imputed_postxform.RenalFailure_v3.txt")

## Carrying forward y1 imputed values to later landmark datasets

In [ ]:
imputed_y5_prep <- ydat5 %>%
    mutate(across(everything(), factorize.i)) %>%
    select(-c(tstart, tstop, endpt, 
              any_of(c(biomarkers_categorize_AoU, paste0(continuous_AoU, "_NA"))), 
              AveragePain7Days, BMI_survey)) %>%
    mutate(t1row = FALSE) 

In [ ]:
imputed_y5 <- imputed_y5_prep %>%
    full_join_quiet(imputed_y1_prexform %>% 
                        select(all_of(names(imputed_y5_prep %>% select(-t1row)))) %>% 
                        mutate(t1row = TRUE) %>%
                        right_join_quiet(ydat5 %>% distinct(person_id))) %>%
    arrange(person_id, desc(t1row)) %>%
    fill(any_of(c(vars_standardize_AoU, "Trig", "WBC", "Platelet", "DCSI")), .direction = "down") %>%
    filter(!t1row) %>%
    ungroup() %>%
    select(-t1row)
nrow(imputed_y5)

In [ ]:
write_to_bucket(imputed_y5,  "landmark_y5_imputed_prexform.RenalFailure_v3.txt")

In [ ]:
imputed_y5_postxform <- imputed_y5 %>%
    mutate(Trig_ln = log(Trig),
          Platelet_ln = log(Platelet),
          WBC_ln = log(WBC)) %>%
    mutate(across(all_of(c(vars_standardize_AoU, "DCSI")), standardize)) %>%
    select(-Trig, -Platelet, -WBC)    %>%
    dplyr::rename(Trig = Trig_ln, Platelet = Platelet_ln, WBC = WBC_ln)

In [ ]:
write_to_bucket(imputed_y5_postxform,  "landmark_y5_imputed_postxform.RenalFailure_v3.txt")

In [ ]:
imputed_y10_prep <- ydat10 %>%
    mutate(across(everything(), factorize.i)) %>%
    select(-c(tstart, tstop, endpt, 
              any_of(c(biomarkers_categorize_AoU, 
                       paste0(continuous_AoU, "_NA"))), 
              AveragePain7Days, BMI_survey)) %>%
    mutate(t1row = FALSE) 

In [ ]:
imputed_y10 <- imputed_y10_prep %>%
    full_join_quiet(imputed_y1_prexform %>% 
                        select(all_of(names(imputed_y10_prep %>% select(-t1row)))) %>% 
                                    mutate(t1row = TRUE) %>% 
                      right_join_quiet(ydat10 %>% distinct(person_id))) %>%
    arrange(person_id, desc(t1row)) %>%
    fill(any_of(c(vars_standardize_AoU, "Trig", "WBC", "Platelet", "DCSI")), .direction = "down") %>%
    filter(!t1row) %>%
    ungroup() %>%
    select(-t1row)
nrow(imputed_y10)

In [ ]:
write_to_bucket(imputed_y10,  "landmark_y10_imputed_prexform.RenalFailure_v3.txt")

In [ ]:
imputed_y10_postxform <- imputed_y10 %>%
    mutate(Trig_ln = log(Trig),
          Platelet_ln = log(Platelet),
          WBC_ln = log(WBC)) %>%
    mutate(across(all_of(c(vars_standardize_AoU, "DCSI")), standardize)) %>%
    select(-Trig, -Platelet, -WBC)    %>%
    dplyr::rename(Trig = Trig_ln, Platelet = Platelet_ln, WBC = WBC_ln)

In [ ]:
write_to_bucket(imputed_y10_postxform,  "landmark_y10_imputed_postxform.RenalFailure_v3.txt")

# Create risk scores based on KDI coefficients

In [ ]:
calc_KDI_riskscore <- function(landmark_data, landmark, type = "scores") {
    coefs <- KDI_coefs %>% 
        filter(landmark_year == landmark) %>%
        distinct(predictor, coef) 
    
    data <- landmark_data %>%
        mutate(
              GenderF = as.numeric(Gender == "Female"),
              RaceBLACK = as.numeric(Race == "Black or African American"),
              SmokingFactorUNKNOWN = as.numeric(SmokingFactor == "UNKNOWN")) %>%
        mutate_if(is.factor, function(x) {as.numeric(as.character(x))} )
    
    data_long <- data %>% select(person_id, all_of(unique(coefs$predictor))) %>%
            pivot_longer(!person_id, names_to = "predictor", values_to = "value") %>%
            full_join_quiet(coefs) %>%
            mutate(score_component = value * coef)
    
    scores <- data_long %>% group_by(person_id) %>%
        summarize(KDI_score = sum(score_component))
    
    if (type == "scores"){
        return(scores)
    } else if (type == "elements"){
        return(data_long) # to examine more closely    
    }
}

Add the risk scores

In [ ]:
imputed_y1_RS <- calc_KDI_riskscore(imputed_y1_postxform, landmark = 1) %>% 
    full_join_quiet(imputed_y1_postxform)
table(is.na(imputed_y1_RS$KDI_score))

In [ ]:
imputed_y5_RS <- calc_KDI_riskscore(imputed_y5_postxform, landmark= 5) %>% 
    full_join_quiet(imputed_y5_postxform)
table(is.na(imputed_y5_RS$KDI_score))

In [ ]:
imputed_y10_RS <- calc_KDI_riskscore(imputed_y10_postxform, landmark = 10) %>% 
    full_join_quiet(imputed_y10_postxform)
table(is.na(imputed_y10_RS$KDI_score))

Checkpoint 3

In [ ]:
now()
ls()
save.image("chkpoint3_latest.RData")

In [ ]:
#load("chkpoint3_latest.RData")

# Evaluate KDI model, get CIFs and Counts

##  Data processing 

In [ ]:
# .._RS objects have the KDI risk score added
imputed_y1_num <- imputed_y1_RS %>% mutate(event_CR = as.numeric(as.character(event_CR))) # not as factor
imputed_y5_num <- imputed_y5_RS %>% mutate(event_CR = as.numeric(as.character(event_CR)))
imputed_y10_num <- imputed_y10_RS %>% mutate(event_CR = as.numeric(as.character(event_CR)))

In [ ]:
#function to process certain fields to same format as KDI predictors, name outcomes as ftime and fstatus
getiL <- function(x, L){
    x %>% 
        mutate(
              GenderF = as.numeric(Gender == "Female"),
              RaceBLACK = as.numeric(Race == "Black or African American"),
              SmokingFactorUNKNOWN = as.numeric(SmokingFactor == "UNKNOWN")) %>%
        mutate(ftime = CR_tte_landmark,
              fstatus = event_CR) %>%
        select(all_of(c("ftime", "fstatus", KDI_coefs %>% filter(landmark_year == 1) %>% .$predictor))
    )
}

In [ ]:
iL1 <- getiL(imputed_y1_num, 1)
iL5 <- getiL(imputed_y5_num, 5)
iL10 <- getiL(imputed_y10_num, 10)

## Fit model in AOU L1 to compare directionality of coefficients to VHA model

In [ ]:
# Convert to numeric input matrix of predictors for L1
i1 <- iL1 %>% select(-ftime, -fstatus) %>% 
    mutate_if(is.factor, function(x){as.numeric(as.character(x))}) %>% 
    as.matrix()

In [ ]:
# model fit at L1 just for comparison, not evaluating performance of this model
set.seed(12345)
AOUFIT <- fastCrr(Crisk(iL1$ftime, iL1$fstatus) ~ i1, 
                  variance = FALSE, returnDataFrame = T,
                  getBreslowJumps = T)

In [ ]:
AOUFIT.coefs <- data.frame(predictor = colnames(i1), coef = AOUFIT$coef)

In [ ]:
write_to_bucket(AOUFIT.coefs, "AOULandmark1FastCrrFit.Coefs_v3.txt")

## Calculate Predicted Probabilities

In [ ]:
KDIpred1 <- predict_prob_calibrated_allH(ftime = imputed_y1_RS$CR_tte_landmark, 
                                         fstatus = imputed_y1_RS$event_CR, 
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y1_RS$KDI_score)

In [ ]:
KDIpred5 <- predict_prob_calibrated_allH(ftime = imputed_y5_RS$CR_tte_landmark, 
                                         fstatus = imputed_y5_RS$event_CR, 
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y5_RS$KDI_score)

In [ ]:
KDIpred10 <- predict_prob_calibrated_allH(ftime = imputed_y10_RS$CR_tte_landmark, 
                                         fstatus = imputed_y10_RS$event_CR, 
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y10_RS$KDI_score)

## Calculate Calibration (Overall Population)

### Metrics

In [ ]:
# Get the observed vs. predicted event rates at each quartile of predicted risk using pseudovalues.
# Also returns the slope and Brier Score. No bootstrapping performed here.

newdata_truth_L1 <- iL1 %>% select(CR_tte_landmark = ftime, event_CR = fstatus)
newdata_truth_L5 <- iL5 %>% select(CR_tte_landmark = ftime, event_CR = fstatus)
newdata_truth_L10 <- iL10 %>% select(CR_tte_landmark = ftime, event_CR = fstatus)

tic("Getting calibration of ESRD-DRS predictions in Overall Population")
KDIcalib_deciles_L1 <- man_cal_quantiles(KDIpred1$pred1, newdata_truth_L1, htime = 1) %>%
    bind_rows(man_cal_quantiles(KDIpred1$pred5, newdata_truth_L1, htime = 5)) %>%
    bind_rows(man_cal_quantiles(KDIpred1$pred10, newdata_truth_L1, htime = 10))

KDIcalib_deciles_L5 <- man_cal_quantiles(KDIpred5$pred1, newdata_truth_L5, htime = 1) %>%
    bind_rows(man_cal_quantiles(KDIpred5$pred5, newdata_truth_L5, htime = 5)) %>%
    bind_rows(man_cal_quantiles(KDIpred5$pred10, newdata_truth_L5, htime = 10))

KDIcalib_deciles_L10 <- man_cal_quantiles(KDIpred10$pred1, newdata_truth_L10, htime = 1) %>%
    bind_rows(man_cal_quantiles(KDIpred10$pred5, newdata_truth_L10, htime = 5)) %>%
    bind_rows(man_cal_quantiles(KDIpred10$pred10, newdata_truth_L10, htime = 10))

toc()

### Plots

In [ ]:
# Data for plotting and exporting
calibplotdat_main <- KDIcalib_deciles_L1 %>%
    mutate(landmark_year = 1) %>%
    rbind(KDIcalib_deciles_L5 %>% mutate(landmark_year = 5) ) %>%
    rbind(KDIcalib_deciles_L10 %>% mutate(landmark_year = 10) ) %>%
    mutate(lbl = paste0("Brier:", signif(BrierScore, 2)),
          lbl = ifelse(pred_grp == 4, lbl, ""),
          lbl2 = paste0("Slope:", signif(Slope, 2)),
          lbl2 = ifelse(pred_grp == 4, lbl2, ""),
          )

In [ ]:
# Line plot
calibplotdat_main %>% 
    ggplot(aes(x=bin_meanpt, y = event_rt, color = factor(htime))) + 
    geom_line() + geom_point() +
    facet_wrap(paste0("Landmark ", landmark_year)~htime, scales = "free") + 
    geom_abline(slope = 1, linetype = "dashed", alpha = 0.4)

In [ ]:
# Bar plot
calibplotdat_main %>% 
    pivot_longer(all_of(c("bin_meanpt", "event_rt")), names_to = "tp", values_to = "val") %>%
    ggplot(aes(x=pred_grp, y = val, fill = tp)) + 
    geom_bar(position = "dodge", stat = "identity") +
    facet_grid(paste0("Landmark ", landmark_year)~htime, scales = "free") + 
    geom_abline(slope = 1, linetype = "dashed", alpha = 0.4)

## Calculate Performance (Overall Population)

### AUPRC

Default of function is 0 bootstraps. We don't do bootstrapping for AUPRC due to time contraints.

In [ ]:
set.seed(12345)

tic("Getting AUPRC for ESRD-DRS L1")
AUPRC.n1KDIbs <- AUPRC_bootstrap(truth = imputed_y1_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y1_num$KDI_score, 
                         htimes = c(1, 5, 10))
AUPRC.n1KDIbs
toc()

In [ ]:
set.seed(12345)

tic("Getting AUPRC for ESRD-DRS L5")
AUPRC.n5KDIbs <- AUPRC_bootstrap(truth = imputed_y5_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y5_num$KDI_score, 
                         htimes = c(1, 5, 10))
AUPRC.n5KDIbs
toc()

In [ ]:
set.seed(12345)

tic("Getting AUPRC for ESRD-DRS L10")
AUPRC.n10KDIbs <- AUPRC_bootstrap(truth = imputed_y10_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y10_num$KDI_score, 
                         htimes = c(1, 5, 10))
AUPRC.n10KDIbs
toc()

### AUC

Get AUC of ESRD-DRS score, get CI via bootstrapping.

In [ ]:
set.seed(12345)

tic("Getting bootstrapped AUC for ESRD-DRS L1")
n1KDIbs <- AUC_bootstrap(imputed_y1_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y1_num$KDI_score, 
                         htimes = c(1, 5, 10))
n1KDIbs
toc()

In [ ]:
set.seed(12345)

tic("Getting bootstrapped AUC for ESRD-DRS L5")
n5KDIbs <- AUC_bootstrap(imputed_y5_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y5_num$KDI_score, 
                         htimes = c(1, 5, 10))
n5KDIbs
toc()

In [ ]:
set.seed(12345)

tic("Getting bootstrapped AUC for ESRD-DRS L10")
n10KDIbs <- AUC_bootstrap(imputed_y10_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y10_num$KDI_score, 
                         htimes = c(1, 5, 10))
n10KDIbs
toc()

## Evaluate Model Performance and Calibration in Subsets

In [ ]:
# Initialize
subset_init_AUC <- data.frame(horiz=0) %>% filter(is.na(horiz))
subset_init_calib <- data.frame(pred_grp=0) %>% filter(is.na(pred_grp))
subset_init_AUPRC <- data.frame(horiz=0) %>% filter(is.na(horiz))

subsetList <- c("DX2000", "DX2010", "DXUNDER65", "DXSENIOR", "EGFRlt60", "EGFRgt60",
                "Hispanic or Latino", "Non-Hispanic", "Male", "Female", 
                 "Black/African American",  "White/Caucasian")
nameKDI <- function(x){paste0("KDI", x)}

# Loop through subsets
tic("Evaluating subsets - KDI models")
for (subset in subsetList) {
    
    # Subset the datasets and add the probabilities (needed for calibration)
    trth1 <- subset_group(imputed_y1_num %>% cbind(KDIpred1 %>% dplyr::rename_with(nameKDI)), 
                          subset, EGFRdat = imputed_y1_prexform)
    trth5 <- subset_group(imputed_y5_num %>% cbind(KDIpred5 %>% dplyr::rename_with(nameKDI)), 
                          subset, EGFRdat = imputed_y5)
    trth10 <- subset_group(imputed_y10_num  %>% cbind(KDIpred10 %>% dplyr::rename_with(nameKDI)), 
                           subset, EGFRdat = imputed_y10) 
    
    # Get bootstrapped AUC in subset
    n1KDIbs.SUBSET <- AUC_bootstrap(trth1, pred = trth1$KDI_score, htimes = c(1, 5, 10))
    n5KDIbs.SUBSET <- AUC_bootstrap(trth5, pred = trth5$KDI_score, htimes = c(1, 5, 10))
    n10KDIbs.SUBSET <- AUC_bootstrap(trth10, pred = trth10$KDI_score, htimes = c(1, 5, 10))
    result_subset_AUC <- rbind(n1KDIbs.SUBSET %>% mutate(landmark = 1),
                          n5KDIbs.SUBSET %>% mutate(landmark = 5),
                          n10KDIbs.SUBSET %>% mutate(landmark = 10)) %>% mutate(subgroup = subset)
    subset_init_AUC <- rbind(subset_init_AUC, result_subset_AUC)
    
    
    # Get AUPRC in subset
    n1KDIbs.SUBSET_AUPRC <- AUPRC_bootstrap(trth1, pred = trth1$KDI_score, htimes = c(1, 5, 10))
    n5KDIbs.SUBSET_AUPRC <- AUPRC_bootstrap(trth5, pred = trth5$KDI_score, htimes = c(1, 5, 10))
    n10KDIbs.SUBSET_AUPRC <- AUPRC_bootstrap(trth10, pred = trth10$KDI_score, htimes = c(1, 5, 10))
    result_subset_AUPRC <- rbind(n1KDIbs.SUBSET_AUPRC %>% mutate(landmark = 1),
                          n5KDIbs.SUBSET_AUPRC %>% mutate(landmark = 5),
                          n10KDIbs.SUBSET_AUPRC %>% mutate(landmark = 10)) %>% mutate(subgroup = subset)
    subset_init_AUPRC <- rbind(subset_init_AUPRC, result_subset_AUPRC)
    
    # Get Calibration metrics in subset
    # skip calibration metrics if < 5 events in subset at L1
    if (sum(trth1$event_CR == 1) >= 5) {
    
        KDIcalib_deciles_L1.SUBSET <- man_cal_quantiles(trth1$KDIpred1, trth1, htime = 1, ntile = 4) %>%
            bind_rows(man_cal_quantiles(trth1$KDIpred5, trth1, htime = 5, ntile = 4)) %>%
            bind_rows(man_cal_quantiles(trth1$KDIpred10, trth1, htime = 10, ntile = 4))

        KDIcalib_deciles_L5.SUBSET <- man_cal_quantiles(trth5$KDIpred1, trth5, htime = 1, ntile = 4) %>%
            bind_rows(man_cal_quantiles(trth5$KDIpred5, trth5, htime = 5, ntile = 4)) %>%
            bind_rows(man_cal_quantiles(trth5$KDIpred10, trth5, htime = 10, ntile = 4))

        KDIcalib_deciles_L10.SUBSET <- man_cal_quantiles(trth10$KDIpred1, trth10, htime = 1, ntile = 4) %>%
            bind_rows(man_cal_quantiles(trth10$KDIpred5, trth10, htime = 5, ntile = 4)) %>%
            bind_rows(man_cal_quantiles(trth10$KDIpred10, trth10, htime = 10, ntile = 4))

        result_subset_calib <- bind_rows(KDIcalib_deciles_L1.SUBSET %>% mutate(landmark_year = 1),
                              KDIcalib_deciles_L5.SUBSET %>% mutate(landmark_year = 5),
                              KDIcalib_deciles_L10.SUBSET %>% mutate(landmark_year = 10)) %>% 
                        mutate(subgroup = subset)
        subset_init_calib <- bind_rows(subset_init_calib, result_subset_calib)
    } else {warning(paste0("There are fewer than 5 events for subgroup at L1: ", subset, "skipping calibration"))}

    # clean up objects
    rm(trth1, trth5, trth10, n1KDIbs.SUBSET, n5KDIbs.SUBSET, n10KDIbs.SUBSET, result_subset_AUC, 
       n1KDIbs.SUBSET_AUPRC, n5KDIbs.SUBSET_AUPRC, n10KDIbs.SUBSET_AUPRC, result_subset_AUPRC,
       KDIcalib_deciles_L1.SUBSET, KDIcalib_deciles_L5.SUBSET, KDIcalib_deciles_L10.SUBSET)
}
toc() 

Combine subset metrics with the overall metrics and write out results

In [ ]:
# calibration
KDImodel_calib <- calibplotdat_main %>% 
    select(-contains("lbl")) %>% 
    mutate(subgroup = "Overall") %>%
    bind_rows(subset_init_calib)

# AUPRC
KDImodel_AUPRC <- AUPRC.n1KDIbs %>% 
    mutate(landmark=1) %>%
    full_join(AUPRC.n5KDIbs %>% mutate(landmark=5)) %>%
    full_join(AUPRC.n10KDIbs %>% mutate(landmark=10)) %>%
    mutate(subgroup = "Overall") %>%
    full_join(subset_init_AUPRC) %>%
    mutate(predictor = "KDI_riskscore")

# AUC
KDImodel_AUC <- rbind(
                  n1KDIbs %>% mutate(landmark=1),
                  n5KDIbs %>% mutate(landmark=5),
                  n10KDIbs %>% mutate(landmark=10)
    ) %>%
    mutate(subgroup = "Overall") %>%
    full_join(subset_init_AUC) %>%
    mutate(predictor = "KDI_riskscore")

In [ ]:
write_to_bucket(KDImodel_calib, "KDImodel_calib.csv")
write_to_bucket(KDImodel_AUPRC, "KDImodel_AUPRC.csv")
write_to_bucket(KDImodel_AUC, "KDImodel_AUC.csv")

## Counts of events/censored/surviving at each time point

### Counts (Overall population)

In [ ]:
#Need N s per group per L per H
tic()
n1KDI <- timeROC(imputed_y1_num$CR_tte_landmark, imputed_y1_num$event_CR, imputed_y1_num$KDI_score, 
        cause = 1, times = c(1, 5, 10), ROC = F, iid = F)
n5KDI <- timeROC(imputed_y5_num$CR_tte_landmark, imputed_y5_num$event_CR, imputed_y5_num$KDI_score, 
        cause = 1, times = c(1, 5, 10), ROC = F, iid = F)
n10KDI <- timeROC(imputed_y10_num$CR_tte_landmark, imputed_y10_num$event_CR, imputed_y10_num$KDI_score, 
        cause = 1, times = c(1, 5, 10), ROC = F, iid = F)
toc()

In [ ]:
time_stats_main <- rbind(n1KDI$Stats %>% data.frame() %>% mutate(landmark = 1) %>% tibble::rownames_to_column("horiz"),
                    n5KDI$Stats %>% data.frame() %>% mutate(landmark = 5) %>% tibble::rownames_to_column("horiz"),
                    n10KDI$Stats %>% data.frame() %>% mutate(landmark = 10) %>% tibble::rownames_to_column("horiz")
                   ) %>%
                mutate(horiz = as.numeric(gsub("t=", "", horiz)))
            
time_stats_main

### Counts (Subsets)

In [ ]:
tic("Getting counts in subsets")
subset_time_stats_init <- data.frame(horiz=0) %>% filter(is.na(horiz))

suppressMessages(
for (subset in subsetList) {
    trth1 <- subset_group(imputed_y1_num, subset, EGFRdat = imputed_y1_prexform)
    trth5 <- subset_group(imputed_y5_num, subset, EGFRdat = imputed_y5)
    trth10 <- subset_group(imputed_y10_num, subset, EGFRdat = imputed_y10) 
    
   
    n1KDI <- timeROC(trth1$CR_tte_landmark, trth1$event_CR, trth1$KDI_score, 
        cause = 1, times = c(1, 5, 10), ROC = F, iid = F)
    n5KDI <- timeROC(trth5$CR_tte_landmark, trth5$event_CR, trth5$KDI_score, 
            cause = 1, times = c(1, 5, 10), ROC = F, iid = F)
    n10KDI <- timeROC(trth10$CR_tte_landmark, trth10$event_CR, trth10$KDI_score, 
            cause = 1, times = c(1, 5, 10), ROC = F, iid = F)
    
    time_stats_subset <- n1KDI$Stats %>% data.frame() %>% mutate(landmark = 1) %>% 
        tibble::rownames_to_column("horiz") %>%
        full_join(n5KDI$Stats %>% data.frame() %>% mutate(landmark = 5) %>% tibble::rownames_to_column("horiz")) %>%
        full_join(n10KDI$Stats %>% data.frame() %>% mutate(landmark = 10) %>% tibble::rownames_to_column("horiz")) %>%
        mutate(horiz = as.numeric(gsub("t=", "", horiz))) %>% 
        mutate(subgroup = subset)
    subset_time_stats_init <- full_join(subset_time_stats_init, time_stats_subset)
    
    rm(trth1, trth5, trth10, n1KDI, n5KDI, n10KDI, time_stats_subset)
}
)

toc()

In [ ]:
# Combine with overall population
time_stats_all <- time_stats_main %>% mutate(subgroup = "Overall") %>%
    full_join(subset_time_stats_init)

In [ ]:
#need to apply masking if counts < 20 and not 0
time_stats_all_masked <- time_stats_all %>%
    mutate(across(Cases:Censored.at.t, mask20))

In [ ]:
# We may need to mask more than 1 category so that counts cannot be derived
time_stats_all_masked %>%
    filter( (as.numeric(grepl("<=20", Cases)) +
            as.numeric(grepl("<=20", survivor.at.t)) + 
            as.numeric(grepl("<=20", Other.events.at.t)) + 
            as.numeric(grepl("<=20", Censored.at.t)) ) == 1) %>%
    head()

In [ ]:
time_stats_all_masked <- time_stats_all_masked %>%
    mutate(Censored.at.t = ifelse(  
            ( as.numeric(grepl("<=20", Cases)) +
              as.numeric(grepl("<=20", survivor.at.t)) + 
              as.numeric(grepl("<=20", Other.events.at.t))
            ) == 1, 
            "Masked", Censored.at.t))

In [ ]:
write_to_bucket(time_stats_all_masked, "landmarking_counts_case_surv_cens_death_v3.csv")

## Cumulative Incidence Function (CIF)

### CIF (Overall population)

In [ ]:
y1dat_cif <- imputed_y1_postxform %>% mutate(event_CR = as.numeric(as.character(event_CR)))
y5dat_cif <- imputed_y5_postxform %>% mutate(event_CR = as.numeric(as.character(event_CR)))
y10dat_cif <- imputed_y10_postxform %>% mutate(event_CR = as.numeric(as.character(event_CR)))

In [ ]:
getcifdata <- function(name) {
    data.frame(
        time = CIFo[[name]]$time,
        CIF = CIFo[[name]]$est,
        var = CIFo[[name]]$var,
        #Subgroup = substr(name, 0, nchar(name) - 2),
        event = substr(name, nchar(name), nchar(name))
    )
}

In [ ]:
# LM1
CIFo <- cuminc(ftime = y1dat_cif$CR_tte_landmark,
              fstatus = y1dat_cif$event_CR)
CIF_df_L1 <- do.call(rbind, lapply(names(CIFo), getcifdata))

In [ ]:
# LM5
CIFo <- cuminc(ftime = y5dat_cif$CR_tte_landmark,
              fstatus = y5dat_cif$event_CR)
CIF_df_L5 <- do.call(rbind, lapply(names(CIFo), getcifdata))  

In [ ]:
# LM10
CIFo <- cuminc(ftime = y10dat_cif$CR_tte_landmark,
              fstatus = y10dat_cif$event_CR)
CIF_df_L10 <- do.call(rbind, lapply(names(CIFo), getcifdata))

In [ ]:
CIF_main <- bind_rows(CIF_df_L1 %>% mutate(landmark=1),
                   CIF_df_L5 %>% mutate(landmark=5), 
                   CIF_df_L10 %>% mutate(landmark=10)) %>% 
    mutate(Event = case_when(event ==1 ~"Renal Failure",
                            event ==2 ~ "Death",
                            TRUE ~ NA))

### CIF (Subsets)

In [ ]:
CIF_subsets <- data.frame(landmark=0) %>% filter(landmark !=0)

for (landmark in c(1, 5, 10)) {
    cifdatL <- get(paste0("y", landmark, "dat_cif")) 
    
    if (landmark == 1) { 
        egfrdatL <- imputed_y1_prexform
    } else { 
        egfrdatL <- get(paste0("imputed_y", landmark))
    }

    for (subset in subsetList) {
        cifdatL.subset <- subset_group(cifdatL, subset, EGFRdat = egfrdatL)
        CIFo <- cuminc(ftime = cifdatL.subset$CR_tte_landmark,
                      fstatus = cifdatL.subset$event_CR)
        CIF_df_i <- do.call(rbind, lapply(names(CIFo), getcifdata))  %>%
            mutate(Event = case_when(event ==1 ~"Renal Failure",
                                    event ==2 ~ "Death",
                                    TRUE ~ NA)) %>%
            mutate(landmark, subset)
        CIF_subsets <- bind_rows(CIF_subsets, CIF_df_i)
        }
}

In [ ]:
CIF_all <- bind_rows(
    CIF_main %>% mutate(subset = "Overall"), 
    CIF_subsets)

In [ ]:
write_to_bucket(CIF_all, "CIF_all_v3.csv")

# Create and evaluate RECODe risk score

## Calculate RECODe score

In [ ]:
# Calculate RECODe score
y1RECODE <- calc_RECODe_riskscore(imputed_y1_prexform %>%
                                 left_join(ydat1 %>% select(person_id, UACR)))
y5RECODE <- calc_RECODe_riskscore(imputed_y5  %>%
                                 left_join(ydat5 %>% select(person_id, UACR)))
y10RECODE <- calc_RECODe_riskscore(imputed_y10  %>%
                                 left_join(ydat10 %>% select(person_id, UACR)))

In [ ]:
# Add the RECODe score to the landmark data objects
imputed_y1_num <- left_join(imputed_y1_num, y1RECODE)
imputed_y5_num <- left_join(imputed_y5_num, y5RECODE)
imputed_y10_num <- left_join(imputed_y10_num, y10RECODE)

## RECODe Calibration

### Overall

#### Use competing risk for S0 - overall population

In [ ]:
# Get predicted recalibrated probabilities
RECODepred1 <- predict_prob_calibrated_allH(ftime = imputed_y1_num$CR_tte_landmark, 
                                            fstatus = imputed_y1_num$event_CR,
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y1_num$RECODe_score)

RECODepred5 <- predict_prob_calibrated_allH(ftime = imputed_y5_num$CR_tte_landmark, 
                                            fstatus = imputed_y5_num$event_CR,
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y5_num$RECODe_score)

RECODepred10 <- predict_prob_calibrated_allH(ftime = imputed_y10_num$CR_tte_landmark, 
                                            fstatus = imputed_y10_num$event_CR,
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y10_num$RECODe_score)

In [ ]:
RECODecalib_deciles_L1 <- man_cal_quantiles(RECODepred1$pred1, newdata_truth_L1, htime = 1) %>%
    bind_rows(man_cal_quantiles(RECODepred1$pred5, newdata_truth_L1, htime = 5)) %>%
    bind_rows(man_cal_quantiles(RECODepred1$pred10, newdata_truth_L1, htime = 10))

RECODecalib_deciles_L5 <- man_cal_quantiles(RECODepred5$pred1, newdata_truth_L5, htime = 1) %>%
    bind_rows(man_cal_quantiles(RECODepred5$pred5, newdata_truth_L5, htime = 5)) %>%
    bind_rows(man_cal_quantiles(RECODepred5$pred10, newdata_truth_L5, htime = 10))

RECODecalib_deciles_L10 <- man_cal_quantiles(RECODepred10$pred1, newdata_truth_L10, htime = 1) %>%
    bind_rows(man_cal_quantiles(RECODepred10$pred5, newdata_truth_L10, htime = 5)) %>%
    bind_rows(man_cal_quantiles(RECODepred10$pred10, newdata_truth_L10, htime = 10))

In [ ]:
calibplotdat_RECODe <- RECODecalib_deciles_L1 %>%
    mutate(landmark = 1) %>%
    rbind(RECODecalib_deciles_L5 %>% mutate(landmark = 5) ) %>%
    rbind(RECODecalib_deciles_L10 %>% mutate(landmark = 10) ) %>%
    mutate(lbl = paste0("Brier:", signif(BrierScore, 3)),
          lbl = ifelse(pred_grp == 4, lbl, ""),
          lbl2 = paste0("Slope:", signif(Slope, 3)),
          lbl2 = ifelse(pred_grp == 4, lbl2, ""),
          ) %>% 
    mutate(calib_method = "fcrr_pseudo")

#### Ignore competing risk for S0 (use cox) - overall population

In [ ]:
tic("Getting predicted probabilities for RECODe (recalibrate with cox)")

RECODepred1_cox <- predict_prob_calibrated_cox(ftime = imputed_y1_num$CR_tte_landmark, 
                                            fstatus = imputed_y1_num$event_CR,
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y1_num$RECODe_score)

RECODepred5_cox <- predict_prob_calibrated_cox(ftime = imputed_y5_num$CR_tte_landmark, 
                                            fstatus = imputed_y5_num$event_CR,
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y5_num$RECODe_score)

RECODepred10_cox <- predict_prob_calibrated_cox(ftime = imputed_y10_num$CR_tte_landmark, 
                                            fstatus = imputed_y10_num$event_CR,
                                              horizons = c(1, 5, 10), 
                                              betax = imputed_y10_num$RECODe_score)
toc()

In [ ]:
Calib_quantiles_riskReg <- get_Calib_quantiles_riskReg(RECODepred1_cox) %>%
    mutate(landmark = 1) %>%
    rbind(get_Calib_quantiles_riskReg(RECODepred5_cox) %>% mutate(landmark = 5)) %>%
    rbind(get_Calib_quantiles_riskReg(RECODepred10_cox) %>% mutate(landmark = 10)) %>%
    select(landmark, htime, everything()) %>% 
    mutate_if(is.numeric, function(x){signif(x, 10)}) #help the sucessful join

Get quantiles and slope, brier score

In [ ]:
RECODecalib_deciles_L1_cox <- man_cal_quantiles_nocmprsk(RECODepred1_cox$pred$pred1, newdata_truth_L1, htime = 1) %>%
    bind_rows(man_cal_quantiles_nocmprsk(RECODepred1_cox$pred$pred5, newdata_truth_L1, htime = 5)) %>%
    bind_rows(man_cal_quantiles_nocmprsk(RECODepred1_cox$pred$pred10, newdata_truth_L1, htime = 10))

RECODecalib_deciles_L5_cox <- man_cal_quantiles_nocmprsk(RECODepred5_cox$pred$pred1, newdata_truth_L5, htime = 1) %>%
    bind_rows(man_cal_quantiles_nocmprsk(RECODepred5_cox$pred$pred5, newdata_truth_L5, htime = 5)) %>%
    bind_rows(man_cal_quantiles_nocmprsk(RECODepred5_cox$pred$pred10, newdata_truth_L5, htime = 10))

RECODecalib_deciles_L10_cox <- man_cal_quantiles_nocmprsk(RECODepred10_cox$pred$pred1, newdata_truth_L10, htime = 1) %>%
    bind_rows(man_cal_quantiles_nocmprsk(RECODepred10_cox$pred$pred5, newdata_truth_L10, htime = 5)) %>%
    bind_rows(man_cal_quantiles_nocmprsk(RECODepred10_cox$pred$pred10, newdata_truth_L10, htime = 10))

In [ ]:
calibplotdat_RECODe_cox <- RECODecalib_deciles_L1_cox %>%
    mutate(landmark = 1) %>%
    rbind(RECODecalib_deciles_L5_cox %>% mutate(landmark = 5) ) %>%
    rbind(RECODecalib_deciles_L10_cox %>% mutate(landmark = 10) ) %>%
    mutate(lbl = paste0("Brier:", signif(BrierScore, 3)),
          lbl = ifelse(pred_grp == 4, lbl, ""),
          lbl2 = paste0("Slope:", signif(Slope, 3)),
          lbl2 = ifelse(pred_grp == 4, lbl2, "")
         ) %>% mutate_if(is.numeric, function(x){signif(x, 10)}) #help the sucessful join

In [ ]:
# Combine the RECODE cox recalibration results
Calibplotdat_RECODE_cox <- calibplotdat_RECODe_cox %>%
    full_join(Calib_quantiles_riskReg) %>% 
    mutate(calib_method = "cox_pseudo")

In [ ]:
#This one has IPCW instead of pseudovalues.  
Calibplotdat_RECODE_cox %>% 
    distinct(landmark, htime, BrierScore_RiskReg) %>%
    pivot_wider(names_from = "htime", values_from = "BrierScore_RiskReg", names_prefix = "H")

In [ ]:
Calibplotdat_RECODE_cox %>% 
    ggplot(aes(x=bin_meanpt, y = event_rt, color = factor(htime))) + 
    geom_line() + geom_point() +
    facet_wrap(paste0("Landmark ", landmark)~htime, scales = "free") + 
    geom_abline(slope = 1, linetype = "dashed", alpha = 0.4) +
    geom_text(aes(label = lbl2, x=bin_meanpt*.5))

#### Combine 2 versions for overall population

In [ ]:
RECODE_recalibration_out <- bind_rows(
    calibplotdat_RECODe, # used competing risks for S0
    Calibplotdat_RECODE_cox # no CR. Use RiskRegression.  Get 2 versions of Brier Score. 
) 

### Subsets

In [ ]:
# Add recode predictions (cmprsk) to objects
nameRECODe <- function(x){paste0("RECODe", x)}
imputed_y1_num <- cbind(imputed_y1_num, RECODepred1 %>% dplyr::rename_with(nameRECODe))
imputed_y5_num <- cbind(imputed_y5_num, RECODepred5 %>% dplyr::rename_with(nameRECODe))
imputed_y10_num <- cbind(imputed_y10_num, RECODepred10 %>% dplyr::rename_with(nameRECODe))

In [ ]:
#Add recode predictions (Cox) to objects
nameRECODeCox <- function(x){paste0("RECODe", x, "cox")}
imputed_y1_num <- cbind(imputed_y1_num, 
                        RECODepred1_cox$pred %>% dplyr::rename_with(nameRECODeCox))                         
imputed_y5_num <- cbind(imputed_y5_num, 
                        RECODepred5_cox$pred %>% dplyr::rename_with(nameRECODeCox))
imputed_y10_num <- cbind(imputed_y10_num, 
                         RECODepred10_cox$pred %>% dplyr::rename_with(nameRECODeCox))

In [ ]:
names(imputed_y1_num %>% select(contains("pred")))

In [ ]:
tic("Evaluating calibration of RECODe score in subsets")
subset_init_RECODe_calib <- data.frame(htime=0) %>% filter(is.na(htime))
    
for (subset in subsetList) {
    
    trth1 <- subset_group(imputed_y1_num, subset, EGFRdat = imputed_y1_prexform)
    trth5 <- subset_group(imputed_y5_num, subset, EGFRdat = imputed_y5)
    trth10 <- subset_group(imputed_y10_num, subset, EGFRdat = imputed_y10) 
    
    # With competing risk S0
    RECODecalib_deciles_L1.SUBSET <- man_cal_quantiles(trth1$RECODepred1, trth1, htime = 1, ntile = 4) %>%
        bind_rows(man_cal_quantiles(trth1$RECODepred5, trth1, htime = 5, ntile = 4)) %>%
        bind_rows(man_cal_quantiles(trth1$RECODepred10, trth1, htime = 10, ntile = 4))

    RECODecalib_deciles_L5.SUBSET <- man_cal_quantiles(trth5$RECODepred1, trth5, htime = 1, ntile = 4) %>%
        bind_rows(man_cal_quantiles(trth5$RECODepred5, trth5, htime = 5, ntile = 4)) %>%
        bind_rows(man_cal_quantiles(trth5$RECODepred10, trth5, htime = 10, ntile = 4))

    RECODecalib_deciles_L10.SUBSET <- man_cal_quantiles(trth10$RECODepred1, trth10, htime = 1, ntile = 4) %>%
        bind_rows(man_cal_quantiles(trth10$RECODepred5, trth10, htime = 5, ntile = 4)) %>%
        bind_rows(man_cal_quantiles(trth10$RECODepred10, trth10, htime = 10, ntile = 4))
    
    # Ignore competing risk for S0 (use cox) 
    RECODecalib_deciles_L1.cox.SUBSET <- man_cal_quantiles_nocmprsk(trth1$RECODepred1cox, trth1, htime = 1, ntile = 4) %>%
        bind_rows(man_cal_quantiles_nocmprsk(trth1$RECODepred5cox, trth1, htime = 5, ntile = 4)) %>%
        bind_rows(man_cal_quantiles_nocmprsk(trth1$RECODepred10cox, trth1, htime = 10, ntile = 4))

    RECODecalib_deciles_L5.cox.SUBSET <- man_cal_quantiles_nocmprsk(trth5$RECODepred1cox, trth5, htime = 1, ntile = 4) %>%
        bind_rows(man_cal_quantiles_nocmprsk(trth5$RECODepred5cox, trth5, htime = 5, ntile = 4)) %>%
        bind_rows(man_cal_quantiles_nocmprsk(trth5$RECODepred10cox, trth5, htime = 10, ntile = 4))

    RECODecalib_deciles_L10.cox.SUBSET <- man_cal_quantiles_nocmprsk(trth10$RECODepred1cox, trth10, htime = 1, ntile = 4) %>%
        bind_rows(man_cal_quantiles_nocmprsk(trth10$RECODepred5cox, trth10, htime = 5, ntile = 4)) %>%
        bind_rows(man_cal_quantiles_nocmprsk(trth10$RECODepred10cox, trth10, htime = 10, ntile = 4))

    # Combine landmark results
    result_subset_RECODe_cox <- bind_rows(RECODecalib_deciles_L1.cox.SUBSET %>% mutate(landmark = 1),
                              RECODecalib_deciles_L5.cox.SUBSET %>% mutate(landmark = 5),
                              RECODecalib_deciles_L10.cox.SUBSET %>% mutate(landmark = 10)) %>% 
        mutate(calib_method = "cox_pseudo") 
    
    result_subset_RECODe_calib <- bind_rows(RECODecalib_deciles_L1.SUBSET %>% mutate(landmark = 1),
                          RECODecalib_deciles_L5.SUBSET %>% mutate(landmark = 5),
                          RECODecalib_deciles_L10.SUBSET %>% mutate(landmark = 10)) %>% 
                    mutate(calib_method = "fcrr_pseudo") %>%
        bind_rows(result_subset_RECODe_cox) %>%
        mutate(subgroup = subset)

    subset_init_RECODe_calib <- bind_rows(subset_init_RECODe_calib, result_subset_RECODe_calib) 
    
    rm(trth1, trth5, trth10, RECODecalib_deciles_L1.SUBSET, RECODecalib_deciles_L5.SUBSET, RECODecalib_deciles_L10.SUBSET,
      RECODecalib_deciles_L1.cox.SUBSET, RECODecalib_deciles_L5.cox.SUBSET, RECODecalib_deciles_L10.cox.SUBSET, 
       result_subset_RECODe_calib)
}
    
toc()

### Combine overall + subsets and write out

In [ ]:
RECODE_recalibration_out_wsubsets <- RECODE_recalibration_out  %>% 
    mutate(subgroup = "Overall") %>%
    bind_rows(subset_init_RECODe_calib)

In [ ]:
write_to_bucket(RECODE_recalibration_out_wsubsets, "RECODE_recalibration_v3.csv")

## RECODe AUC

### Overall population

In [ ]:
n1RCbs <- AUC_bootstrap(imputed_y1_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y1_num$RECODe_score, 
                         htimes = c(1, 5, 10))
n5RCbs <- AUC_bootstrap(imputed_y5_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y5_num$RECODe_score, 
                         htimes = c(1, 5, 10))
n10RCbs <- AUC_bootstrap(imputed_y10_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y10_num$RECODe_score, 
                         htimes = c(1, 5, 10))

In [ ]:
RECODe_AUC_main <- rbind(n1RCbs %>% mutate(landmark=1),
                         n5RCbs %>% mutate(landmark=5),
                         n10RCbs %>% mutate(landmark=10)
                        )

### Subsets

In [ ]:
tic("Evaluating performance of RECODe score in subsets")
subset_init_recode <- data.frame(horiz=0) %>% filter(is.na(horiz))
    
for (subset in subsetList) {
    trth1 <- subset_group(imputed_y1_num, subset, EGFRdat = imputed_y1_prexform)
    trth5 <- subset_group(imputed_y5_num, subset, EGFRdat = imputed_y5)
    trth10 <- subset_group(imputed_y10_num, subset, EGFRdat = imputed_y10) 
    
    n1RCbs.SUBSET <- AUC_bootstrap(trth1, pred = trth1$RECODe_score, htimes = c(1, 5, 10))
    n5RCbs.SUBSET <- AUC_bootstrap(trth5, pred = trth5$RECODe_score, htimes = c(1, 5, 10))
    n10RCbs.SUBSET <- AUC_bootstrap(trth10, pred = trth10$RECODe_score, htimes = c(1, 5, 10))
    
    result_subset_recode <- rbind(n1RCbs.SUBSET %>% mutate(landmark = 1),
                          n5RCbs.SUBSET %>% mutate(landmark = 5),
                          n10RCbs.SUBSET %>% mutate(landmark = 10)) %>% mutate(subgroup = subset)
    subset_init_recode <- full_join_quiet(subset_init_recode, result_subset_recode) 
    
    rm(trth1, trth5, trth10, n1RCbs.SUBSET, n5RCbs.SUBSET, n10RCbs.SUBSET, result_subset_recode)
}
toc()

In [ ]:
# Combine subset results with overall and write out
RECODe_AUC_all <- bind_rows(RECODe_AUC_main,
                            subset_init_recode) %>% 
    mutate(predictor = "RECODe_score")

In [ ]:
write_to_bucket(RECODe_AUC_all, "RECODe_AUC.csv")

## RECODe AUPRC

### Overall population

In [ ]:
set.seed(12345)

AUPRC.n1RCbs <- AUPRC_bootstrap(truth = imputed_y1_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y1_num$RECODe_score, 
                         htimes = c(1, 5, 10))
AUPRC.n5RCbs <- AUPRC_bootstrap(truth = imputed_y5_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y5_num$RECODe_score, 
                         htimes = c(1, 5, 10))
AUPRC.n10RCbs <- AUPRC_bootstrap(truth = imputed_y10_num %>% select(CR_tte_landmark, event_CR), 
                         pred = imputed_y10_num$RECODe_score, 
                         htimes = c(1, 5, 10))

In [ ]:
RECODE_AUPRC_main <- AUPRC.n1RCbs %>% 
    mutate(landmark=1) %>%
    full_join(AUPRC.n5RCbs %>% mutate(landmark=5)) %>%
    full_join(AUPRC.n10RCbs %>% mutate(landmark=10))

### Subsets

In [ ]:
#initialize
subset_init_AUPRC_recode <- data.frame(horiz=0) %>% filter(is.na(horiz))

tic("Evaluating AUPRC in subsets - RECODe")
for (subset in subsetList) {
    
    trth1 <- subset_group(imputed_y1_num, subset, EGFRdat = imputed_y1_prexform)
    trth5 <- subset_group(imputed_y5_num, subset, EGFRdat = imputed_y5)
    trth10 <- subset_group(imputed_y10_num, subset, EGFRdat = imputed_y10) 
    
    n1RCbs.SUBSET_AUPRC <- AUPRC_bootstrap(trth1, pred = trth1$RECODe_score, htimes = c(1, 5, 10))
    n5RCbs.SUBSET_AUPRC <- AUPRC_bootstrap(trth5, pred = trth5$RECODe_score, htimes = c(1, 5, 10))
    n10RCbs.SUBSET_AUPRC <- AUPRC_bootstrap(trth10, pred = trth10$RECODe_score, htimes = c(1, 5, 10))
    
    result_subset_AUPRC_recode <- rbind(n1RCbs.SUBSET_AUPRC %>% mutate(landmark = 1),
                          n5RCbs.SUBSET_AUPRC %>% mutate(landmark = 5),
                          n10RCbs.SUBSET_AUPRC %>% mutate(landmark = 10)) %>% mutate(subgroup = subset)
    subset_init_AUPRC_recode <- rbind(subset_init_AUPRC_recode, result_subset_AUPRC_recode)
    
    rm(trth1, trth5, trth10, n1RCbs.SUBSET_AUPRC, n5RCbs.SUBSET_AUPRC, n10RCbs.SUBSET_AUPRC, result_subset_AUPRC_recode)
}
toc() 

In [ ]:
RECODE_AUPRC_all <- RECODE_AUPRC_main %>%
    mutate(subgroup = "Overall") %>%
    full_join(subset_init_AUPRC_recode) %>%
    mutate(predictor = "RECODe_score")

In [ ]:
write_to_bucket(RECODE_AUPRC_all, "RECODE_AUPRC.csv")

# Save image, Cleanup

In [ ]:
toc()

In [ ]:
# Note end time before saving image
now()
today_string <- gsub("-", "", lubridate::today())
today_string
ls()
save.image(paste0("landmarkeval_endpt_", today_string, ".RData"))

In [ ]:
#load("landmarkeval_endpt_XXX.RData")

In [ ]:
#rm(list = ls()); gc()